# MOE

## 1.核心思想
- Dense (稠密) 模型： 在标准的 Transformer 中，每一个 Token 都必须经过全网络的所有参数（比如 70B 的 LLaMA）。这导致随着模型变大，推理和训练的计算量呈线性爆炸。
- MoE ：稀疏激活 (Sparse Activation) 将原来大规模的 MLP 层，切分成N个小型的独立 MLP（称为 Expert，专家）。 对于每一个输入的 Token，通过一个非常轻量的 Router (门控网络) 决定它该去请教哪K个专家（通常K=2）。这样，即便总参数量有 8x7B=56B，实际每个 Token 只激活了 2x7B=14B 的参数。计算量骤降，而知识容量剧增。

## 2. 实现流程
实现顺序是 `router_logits -> 全局 softmax -> top-k -> 重归一化 -> sparse dispatch`。

1. **门控网络（Gating / Router）**：给定输入 Token 的特征 $x \in \mathbb{R}^d$，先通过一个线性层得到每个专家的打分，也叫路由分数：

    $$
    h = x W_{gate} (h\in \mathbb{R}^E)
    $$
    其中，$E$ 表示专家总数，比如 8 个专家。

2. **全局归一化与 Top-K 选择**：初学者常见的错误是先做 Top-K，再做 Softmax。更标准的做法是先对所有专家分数做全局 Softmax，得到每个专家的概率：

    $$
    p = \mathrm{Softmax}(h)  (p\in \mathbb{R}^E)
    $$

    然后从中选出概率最大的 $K$ 个专家：

    $$
    P_{topk},\ idx_{topk} = \mathrm{TopK}(p, K)
    $$

3. **局部重归一化**：由于只保留了 Top-K 的一部分概率，这些概率之和通常不再等于 1。为了保证后续加权更稳定，需要重新归一化：

    $$
    \tilde{P}_i = \frac{P_i}{\sum_{j \in topK} P_j}
    $$

4. **最终输出融合**：Token 会分别送入这 $K$ 个专家，最后按照新的权重进行加权求和：

    $$
    y = \sum_{i \in topK} \tilde{P}_i \cdot \mathrm{Expert}_i(x)
    $$

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class TopKRouter(nn.Module):
    def __init__(self, hidden_size: int, num_experts: int, top_k: int):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # 定义门控层，将隐藏状态映射到专家数量的得分
        self.gate = nn.Linear(hidden_size, num_experts, bias=False)

    def forward(self, hidden_states: torch.Tensor):
        """
        Args:
            hidden_states: [batch_size, seq_len, hidden_size]
        Returns:
            routing_weights: 形状 [batch_size * seq_len, top_k]，表示选中的专家的权重 (重归一化后)
            selected_experts: 形状 [batch_size * seq_len, top_k]，表示选中的专家索引
        """
        batch_size, seq_len, hidden_size = hidden_states.shape
        # 展平输入
        hidden_states = hidden_states.view(-1, hidden_size)
        
        # 1. 计算 logits 得分
        router_logits = self.gate(hidden_states)
        
        # ==========================================
        # TODO 1: 对全量 Logits 进行 Softmax 获取所有专家的概率分布
        # 提示: 强制使用 FP32 以防止精度溢出 (router_logits.float())
        # ==========================================
        # routing_probs = ???
        routing_probs = F.softmax(router_logits.float(), dim=-1)

        # ==========================================
        # TODO 2: 从概率分布中截取 Top-K 最大的概率 (routing_weights) 及其索引 (selected_experts)
        # ==========================================
        # routing_weights, selected_experts = ???
        routing_weights, selected_experts = torch.topk(routing_probs, self.top_k, dim=-1)
        
        # ==========================================
        # TODO 3: 对截取后的 routing_weights 进行重归一化 (Re-normalize)
        # 提示: 让这 K 个专家的概率按比例放大，使其加和等于 1
        # ==========================================
        # routing_weights = ???
        routing_weights = routing_weights / routing_weights.sum(dim=-1, keepdim=True)
                                                                                                    
        
        # 恢复到原始数据类型
        routing_weights = routing_weights.to(hidden_states.dtype)
        
        return routing_weights, selected_experts

# 为了验证 Router 能正确工作，我们写一个极简的 MoE 聚合层
class SparseMoEBlock(nn.Module):
    def __init__(self, hidden_size: int, num_experts: int, top_k: int):
        super().__init__()
        self.router = TopKRouter(hidden_size, num_experts, top_k)
        # 极简模拟 Expert (真实的 Expert 通常是 SwiGLU MLP)
        self.experts = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(num_experts)])
        
    def forward(self, hidden_states: torch.Tensor):
        batch_size, seq_len, hidden_size = hidden_states.shape
        routing_weights, selected_experts = self.router(hidden_states)
        
        final_hidden_states = torch.zeros(
            (batch_size * seq_len, hidden_size), 
            dtype=hidden_states.dtype, 
            device=hidden_states.device
        )
        flat_hidden_states = hidden_states.view(-1, hidden_size)
        
        # 工业界(vLLM/Megatron)会通过 Token Sorting (索引排序) 汇聚同专家的Token，
        # 这里为便于理解核心算法逻辑，使用 For 循环遍历被选中的 Expert
        for expert_idx, expert in enumerate(self.experts):
            token_idx, kth_expert = torch.where(selected_experts == expert_idx)
            if token_idx.shape[0] > 0:
                current_state = flat_hidden_states[token_idx]
                current_output = expert(current_state)
                current_weight = routing_weights[token_idx, kth_expert].unsqueeze(-1)
                final_hidden_states[token_idx] += current_output * current_weight
                
        return final_hidden_states.view(batch_size, seq_len, hidden_size)

In [3]:
# 运行此单元格以测试你的实现
def test_moe_router():
    try:
        torch.manual_seed(42)
        batch_size, seq_len, hidden_size = 2, 4, 16
        num_experts, top_k = 8, 2
        
        moe = SparseMoEBlock(hidden_size, num_experts, top_k)
        x = torch.randn(batch_size, seq_len, hidden_size)
        
        # 1. 验证输出形状
        out = moe(x)
        assert out.shape == x.shape, "MoE 聚合后的输出形状不匹配！"
        
        # 2. 验证 Router 行为
        weights, indices = moe.router(x)
        assert weights.shape == (batch_size * seq_len, top_k), "权重形状不等于 [num_tokens, top_k]！"
        assert indices.shape == (batch_size * seq_len, top_k), "索引形状不等于 [num_tokens, top_k]！"
        
        # 验证重归一化是否正确 (每一行的和应非常接近 1)
        assert torch.allclose(weights.sum(dim=-1), torch.ones(batch_size * seq_len, dtype=weights.dtype)), "重归一化失败：Top-K 权重之和不等于 1！"
        
        # 3. 验证专家索引合法性
        assert torch.all((indices >= 0) & (indices < num_experts)), "挑选的专家索引越界！"
        
        print("\n✅ All Tests Passed! MoE Top-K Router 和稀疏聚合逻辑验证通过。")
        
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError) as e:
        print("代码可能未完成，导致变量未定义" if isinstance(e, NameError) else "代码可能未完成，导致了类型错误")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except Exception as e:
        print(f"❌ 发生未知异常: {e}")
        raise

test_moe_router()


✅ All Tests Passed! MoE Top-K Router 和稀疏聚合逻辑验证通过。


## 3. 负载均衡损失

为了让 Token 尽量均匀地分配到不同专家上，MoE 通常会额外加入一个 **负载均衡损失**（Load Balancing Loss）。
这个损失的作用很简单：**不让某几个专家一直被“挤爆”，也不让某些专家长期闲着**。

Mixtral / Switch Transformer 常用的辅助损失可以写成：

$$
\mathcal{L}_{\text{aux}} = \alpha \cdot E \sum_{i=1}^{E} f_i P_i
$$

其中：

- $E$：专家总数。
- $\alpha$：辅助损失权重，通常取很小的值，比如 $0.01$。
- $f_i$：第 $i$ 个专家的实际分配频率，也就是它被选中的 Token 占比。
- $P_i$：第 $i$ 个专家的平均路由概率，也就是 Router 对这个专家整体偏好的强弱。

在 Top-K 路由（K≥1）下，本教程统一定义为：

$$
f_i = \frac{1}{T·K} \sum_{t=1}^{T} \mathbb{1}_{(i \in topK_(p_t) )}
$$

$$
P_i = \frac{1}{T} \sum_{t=1}^{T} p_{t,i}
$$

其中：
- $T$：当前批次中参与路由统计的 token 总数，通常为T = batch_size x sequence_length。
- $K$：每个 token 选择的专家数（通常为 2 或 4）。
- $p_{t,i}$：第 $t$ 个 token 在 全部 $E$ 个专家上 的路由概率分布向量，满足 $\sum_{i=1}^{E} p_{t,i} = 1$。其中 $p_{t,i}$ 表示 token $t$ 分配给专家 $i$ 的 Softmax 概率；$topK_(p_t)$ 表示从该向量中选出的前 $K$ 个专家索引。

这里可以把它理解成两个视角的乘积：$P_i$ 描述路由器“想把 token 分给谁”，$f_i$ 描述实际“分给了谁”；只有两者同时偏向同一批专家时，loss 才会明显上升，从而把路由从塌缩状态拉回均匀状态。

In [4]:
def compute_load_balancing_loss(
    routing_weights: torch.Tensor, 
    selected_experts: torch.Tensor, 
    num_experts: int, 
    top_k: int,
    alpha: float = 0.01
):
    """
    计算 MoE 的负载均衡辅助损失（支持 Top-K 路由）
    
    Args:
        routing_weights: [batch_size * seq_len, top_k]，每个 token 选中的 K 个专家的权重（已归一化）
        selected_experts: [batch_size * seq_len, top_k]，每个 token 选中的 K 个专家的索引
        num_experts: 专家总数 E
        top_k: 每个 token 选择的专家数量 K
        alpha: 损失权重系数
    
    Returns:
        aux_loss: 标量，负载均衡损失
    """
    batch_size_x_seq_len, _ = selected_experts.shape
    total_tokens = batch_size_x_seq_len
    
    # ==========================================
    # 先统计每个专家拿到的平均路由概率，再做后续归一化。
    # TODO 1: 计算 P_i（每个专家的平均路由概率得分）
    # ==========================================
    # P_i = ???
    P_i = torch.zeros(num_experts, dtype=routing_weights.dtype, device=routing_weights.device)
    P_i.scatter_add_(0, selected_experts.flatten(), routing_weights.flatten())
    P_i = P_i / total_tokens
    
    # ==========================================
    # 再统计每个专家实际被选中的次数，形成分配比例。
    # TODO 2: 计算 f_i（每个专家实际分到的 Token 比例）
    # ==========================================
    # expert_mask = ???
    # tokens_per_expert = ???
    # f_i = ???
    expert_mask = F.one_hot(selected_experts, num_classes=num_experts)
    tokens_per_expert = expert_mask.sum(dim=(0, 1)).float()
    f_i = tokens_per_expert / (total_tokens * top_k)
    
    # ==========================================
    # 最后把两种视角的分布点乘，得到负载均衡损失。
    # TODO 3: 计算最终的 auxiliary loss
    # ==========================================
    # aux_loss = ???
    aux_loss = alpha * num_experts * (f_i * P_i).sum()

    return aux_loss

In [5]:
# 测试你的实现
def test_aux_loss():
    try:
        torch.manual_seed(42)
        num_experts = 8
        top_k = 2
        num_tokens = 1000
        alpha = 0.01
        
        # 模拟路由结果
        # 1. 极度不均衡：所有 token 都选专家 0 和 1
        bad_selected = torch.zeros(num_tokens, top_k, dtype=torch.long)
        bad_selected[:, 0] = 0
        bad_selected[:, 1] = 1
        bad_weights = torch.ones(num_tokens, top_k) / top_k
        
        loss_bad = compute_load_balancing_loss(bad_weights, bad_selected, num_experts, top_k, alpha)
        
        # 2. 绝对均匀：token 均匀分配给所有专家
        good_selected = torch.zeros(num_tokens, top_k, dtype=torch.long)
        for i in range(num_tokens):
            good_selected[i, 0] = (i * 2) % num_experts
            good_selected[i, 1] = (i * 2 + 1) % num_experts
        good_weights = torch.ones(num_tokens, top_k) / top_k
        
        loss_good = compute_load_balancing_loss(good_weights, good_selected, num_experts, top_k, alpha)
        
        print(f"极度不均衡的 Loss: {loss_bad.item():.4f}")
        print(f"绝对均匀的 Loss  : {loss_good.item():.4f}")
        
        # 理论最小值验证（基于当前 Top-K 定义）
        # 均匀时：f_i = (T*K/E) / (T*K) = 1/E；P_i = 1/E
        # => sum(f_i * P_i) = E*(1/E)*(1/E) = 1/E
        # => aux_loss = alpha * E * (1/E) = alpha
        expected_min = alpha  # 0.01
        assert torch.allclose(loss_good, torch.tensor(expected_min), atol=1e-4), f"理论最小 Loss 计算错误！期望 {expected_min:.4f}，实际 {loss_good.item():.4f}"
        assert loss_bad > loss_good * 2, "惩罚项没有对不均衡分布产生足够大的 Loss！"
        
        print("\n✅ All Tests Passed! 你成功掌握了 Mixtral / DeepSeek 的防崩塌核心技术！")
        
    except NotImplementedError:
        print("请先完成 TODO 代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError) as e:
        print("代码可能未完成，导致变量未定义" if isinstance(e, NameError) else "代码可能未完成，导致了类型错误")
        raise NotImplementedError("请先完成 TODO 代码！") from e
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
        raise NotImplementedError("请先完成 TODO 代码！") from e
    except Exception as e:
        print(f"❌ 测试失败: {e}")
        raise

test_aux_loss()

极度不均衡的 Loss: 0.0400
绝对均匀的 Loss  : 0.0100

✅ All Tests Passed! 你成功掌握了 Mixtral / DeepSeek 的防崩塌核心技术！
